In [1]:
import yfinance as yf
import pandas as pd
import datetime as dt
import numpy as np

# database
def get_data_tickers():
    url = "https://raw.githubusercontent.com/heyhanief/IDX-Data/main/Kompas100"
    try:
        df = pd.read_csv(url, header=None)
        tickers = df[0].astype(str).tolist()
        tickers = [t if t.endswith(".JK") else f"{t}.JK" for t in tickers]
        print(f"Successful to import {len(tickers)} tickers")
        return tickers
    except Exception as e:
        print(f"Failed to import tickers: {e}")
        return []
    
tickers = get_data_tickers()
print(f"Analyzing {len(tickers)} tickers...")

end = dt.date.today()
start = end - dt.timedelta(days=200)

# Fix: explicitly set auto_adjust parameter
data = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
close = data["Close"]
high = data["High"]
low = data["Low"]
volume = data["Volume"]

# Moving Averages
ma21 = close.rolling(21).mean()
ma50 = close.rolling(50).mean()

# ADR (Average Daily Range) - 20-day period
daily_range = ((high - low) / close) * 100  # Daily range in percentage
adr_20 = daily_range.rolling(20).mean()  # 20-day average daily range

today = close.index[-1]
yesterday = close.index[-2]

results = []

for ticker in close.columns:
    price = close[ticker].loc[today]
    vol = volume[ticker].loc[today]
    if pd.isna(price) or pd.isna(vol):
        continue

    # MA distance
    ma_dist = ma21[ticker].loc[today] - ma50[ticker].loc[today]
    ma_dist_yesterday = ma21[ticker].loc[yesterday] - ma50[ticker].loc[yesterday]
    ma_gap_percent = ma_dist / ma50[ticker].loc[today] * 100  # Gap MA21-MA50 in %
    price_ma21_gap = abs(price - ma21[ticker].loc[today]) / ma21[ticker].loc[today] * 100  # Gap Price-MA21 in %

    # ADR condition
    adr_value = adr_20[ticker].loc[today] if not pd.isna(adr_20[ticker].loc[today]) else 0

    # Conditions
    cond1 = price > ma21[ticker].loc[today]       # Price > MA21
    cond2 = price > ma50[ticker].loc[today]       # Price > MA50
    cond3 = ma21[ticker].loc[today] > ma50[ticker].loc[today]
    cond4 = ma21[ticker].loc[today] > ma21[ticker].loc[yesterday]  # MA21 rising
    cond5 = ma_dist > 0 and ma_dist > ma_dist_yesterday  # MA21 above MA50 & gap increasing
    cond6 = price >= 100 and price <= 2500         # Harga saham > 100 dan < 2500
    cond7 = vol >= 50000        # Volume >= 50k
    cond8 = adr_value > 3.0     # ADR > 3%

    if cond1 and cond2 and cond3 and cond4 and cond5 and cond6 and cond7 and cond8:
        results.append({
            'ticker': ticker,
            'price': price,
            'ma21_ma50_gap': ma_gap_percent,
            'price_ma21_gap': price_ma21_gap,
            'adr': adr_value
        })

# Sort by ADR (highest first)
results.sort(key=lambda x: x['adr'], reverse=True)

print("\nPotential Up List (filtered with volume, price, MA gaps, and ADR > 3%):")
print(f"{'Ticker':<12} {'Price':>8} {'MA21-MA50 Gap':>14} {'Price-MA21 Gap':>15} {'ADR':>8}")
print("-" * 70)

for r in results:
    print(f"{r['ticker']:<12} {r['price']:>8.0f} {r['ma21_ma50_gap']:>13.2f}% {r['price_ma21_gap']:>14.2f}% {r['adr']:>7.2f}%")

print(f"\nTotal stocks found: {len(results)}")

Successful to import 100 tickers
Analyzing 100 tickers...

Potential Up List (filtered with volume, price, MA gaps, and ADR > 3%):
Ticker          Price  MA21-MA50 Gap  Price-MA21 Gap      ADR
----------------------------------------------------------------------
ELSA.JK           675          7.31%          18.03%    6.65%
HMSP.JK           810          0.46%           3.44%    5.05%
PGAS.JK          2180          7.18%           5.92%    4.48%
MYOR.JK          2360          0.94%           8.40%    3.58%
BBTN.JK          1220          1.44%           1.93%    3.21%

Total stocks found: 5
